In [1]:
# Library
import pandas as pd
import sqlite3
import os

In [2]:
# Importing datasets using pandas
retail_raw = pd.read_csv('data.nosync/online_retail_II.zip',compression="zip")
print("--- Overview ---")
print(retail_raw.info())
print("--- Missing Value ---")
print(retail_raw.isna().sum())

--- Overview ---
<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  str    
 1   StockCode    1067371 non-null  str    
 2   Description  1062989 non-null  str    
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  str    
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 136.6 MB
None
--- Missing Value ---
Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64


## Data Ingestion (Bronze Layer)

In [3]:
def cleaning(df):
    df_clean = df.copy()
    # 1. Standardizing  date-time format and column names to snake case
    df_clean.columns = df_clean.columns.str.strip().str.lower().str.replace(" ","_")
    df_clean['invoicedate'] = pd.to_datetime(df_clean['invoicedate'],errors="coerce")
    return df_clean
retail_bronze = cleaning(retail_raw)

## Loading to database

In [4]:
# 1. Define path
db_path = 'data.nosync/online_retail.db'
with sqlite3.connect(db_path) as conn:
    retail_bronze.to_sql("retail_bronze", conn, if_exists="replace",index=False)
%load_ext sql
%sql sqlite:///data.nosync/online_retail.db
print("--- Data saved to retail_bronze table. ---")

Connecting to 'sqlite:///data.nosync/online_retail.db'

--- Data saved to retail_bronze table. ---


## Silver Layer

### Data Cleaning

In [5]:
%%sql
WITH de_duplicated AS (
    SELECT *, ROW_NUMBER() over (PARTITION BY invoice,stockcode,description,
                         quantity,invoicedate,price,customer_id,
                        country ORDER BY invoicedate DESC) pk
    FROM retail_bronze
)
SELECT count(*) AS `Number of duplicates` 
FROM de_duplicated 
WHERE pk>1

Running query in 'sqlite:///data.nosync/online_retail.db'

Number of duplicates
34335


In [6]:
%%sql
SELECT count(*) AS `Total Rows`,
       SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS `Missing Value in customer_id`,
       SUM(CASE WHEN description IS NULL THEN 1 ELSE 0 END) AS `Missing Value in description`
FROM retail_bronze;

Running query in 'sqlite:///data.nosync/online_retail.db'

Total Rows,Missing Value in customer_id,Missing Value in description
1067371,243007,4382


In [7]:
%%sql
SELECT COUNT(quantity) AS `Total number of reuturned Items`, ABS(sum(price*quantity)) `Total Return Value`
FROM retail_bronze
WHERE quantity<=0

Running query in 'sqlite:///data.nosync/online_retail.db'

Total number of reuturned Items,Total Return Value
22950,1527041.43


In [8]:
%%sql
DROP TABLE if EXISTS retail_silver;
CREATE TABLE retail_silver (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    invoice	TEXT,
    stockcode TEXT,
    description	TEXT,
    quantity INTEGER,
    invoicedate TIMESTAMP,
    unit_price REAL,
    customer_id REAL,
    country TEXT,
    revenue REAL
);
INSERT INTO retail_silver (
    invoice,stockcode,description,quantity,invoicedate,
    unit_price,customer_id,country,revenue
)
WITH de_duplicated AS (
    SELECT *, ROW_NUMBER() over (PARTITION BY invoice,stockcode,description,
                         quantity,invoicedate,price,customer_id,
                        country ORDER BY invoicedate DESC) pk
    FROM retail_bronze
)
SELECT invoice,stockcode,COALESCE(description,"unknown"),quantity,
       invoicedate,price AS unit_price,customer_id,
       country,price*quantity AS revenue
FROM de_duplicated 
WHERE pk=1 
      AND quantity>0 
      AND customer_id IS NOT NULL
      AND unit_price > 0;

Running query in 'sqlite:///data.nosync/online_retail.db'

779425 rows affected.

++
||
++
++

In [9]:
%%sql
SELECT (SELECT COUNT(*) FROM retail_bronze) AS bronze_count,
       (SELECT COUNT(*) FROM retail_silver) AS silver_count,
       ((SELECT COUNT(*) FROM retail_bronze)- (SELECT COUNT(*) FROM retail_silver))*100/
    (SELECT COUNT(*) FROM retail_bronze) AS `% Rows dropped`

Running query in 'sqlite:///data.nosync/online_retail.db'

bronze_count,silver_count,% Rows dropped
1067371,779425,26


## Gold Layer

In [36]:
%%sql
DROP VIEW IF EXISTS gold_cohort_analysis;
CREATE VIEW gold_cohort_analysis AS
SELECT DATE(invoicedate,"start of month") AS `Invoice_Month`, MIN (DATE(invoicedate,"start of month")) over 
    (partition by customer_id) AS `Cohort_Month`
FROM retail_silver;    


Running query in 'sqlite:///data.nosync/online_retail.db'

++
||
++
++

In [37]:
%%sql
SELECT * FROM gold_cohort_analysis;

Running query in 'sqlite:///data.nosync/online_retail.db'

Invoice_Month,Cohort_Month
2009-12-01,2009-12-01
2009-12-01,2009-12-01
2009-12-01,2009-12-01
2009-12-01,2009-12-01
2009-12-01,2009-12-01
2010-01-01,2009-12-01
2010-01-01,2009-12-01
2010-01-01,2009-12-01
2010-01-01,2009-12-01
2010-03-01,2009-12-01
